# 🤖 04 · Criando um Genie Agent que fala a língua do negócio

**Objetivo:** criar um **Genie Agent** (antigo Genie Space) para o time comercial/CX da TelcoBR e mostrar, na prática, como **instruções** e **consultas SQL de exemplo** transformam respostas "mais ou menos" em respostas **certas e consistentes**.

⏱️ **Duração:** 45–60 min

### 🧭 A dinâmica do exercício: *Antes* ➜ *Ensinar* ➜ *Depois*
Para cada KPI vamos:
1. ❓ **Perguntar** ao Genie sem nenhum contexto e ver o que ele responde.
2. 🔍 **Comparar** com o que a área de negócio considera correto (células deste notebook).
3. 🧑‍🏫 **Ensinar** o Genie — com uma **instrução em texto** e/ou uma **consulta SQL de exemplo**.
4. ✅ **Perguntar de novo** (em um chat novo) e ver a resposta certa.

> 🎯 **Mensagem-chave:** o Genie é muito bom em SQL, mas **não conhece as regras da sua empresa**. Quem ensina é o time de negócio — em português.

### ⚙️ Configuração — rode esta célula primeiro (ou clique em **Executar tudo**)
Use o **mesmo Catálogo e Schema** do notebook 01 (campos no topo do notebook; padrão no Free Edition: `workspace` / `workshop_negocio`).
As caixas roxas 📋 ao longo do notebook mostram os prompts **já com o nome das suas tabelas** — é só copiar e colar.

In [0]:
import html
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("schema", "workshop_negocio", "Schema")
CATALOGO, SCHEMA = dbutils.widgets.get("catalogo").strip(), dbutils.widgets.get("schema").strip()
T = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

def prompt(texto, titulo="📋 Copie e cole"):
    """Mostra um prompt/SQL pronto para copiar, trocando {T} por catálogo.schema."""
    corpo = html.escape(texto.strip().replace("{T}", T))
    displayHTML(f"""<div style="font-family:sans-serif;font-size:12px;font-weight:bold;color:#6929C4;margin:2px 0">{titulo}</div>
<pre style="background:#F4F0FB;border-left:4px solid #6929C4;padding:10px 14px;white-space:pre-wrap;font-size:13px;margin:0">{corpo}</pre>""")

prompt(f"Catálogo: {CATALOGO}\nSchema:   {SCHEMA}\nSuas tabelas: {T}.<tabela>", "✅ Configuração ativa")

---
## 🚀 Passo 1 · Crie o Genie Agent

1. Menu lateral → **Genie** → **+ Novo** (Genie Agent).
2. **Fontes de dados** → navegue até o **seu catálogo → `workshop_negocio`** (veja a caixa ✅ Configuração ativa acima) e adicione:
   `clientes`, `contratos`, `planos`, `lojas`, `base_mensal`, `faturamento_mensal`, `atendimentos`.
3. **Nome:** `Assistente Comercial TelcoBR - <seu nome>`
4. **Descrição** (cole):
```
Assistente para o time Comercial e de Experiência do Cliente da TelcoBR. Responde perguntas sobre receita, base de clientes, churn (cancelamentos), ARPU, inadimplência, vendas por canal e atendimento/NPS. Dados mensais de nov/2024 a set/2026.
```
5. **Perguntas de exemplo** (aparecem como sugestão para quem abrir o agente):
```
Qual foi a receita total por mês em 2026?
Qual o NPS por canal de atendimento?
Como evoluiu o churn mensal do Pós-pago depois do reajuste?
Quais os 5 principais motivos de cancelamento?
```
6. Escolha o **SQL Warehouse** sugerido e salve.

### 🔥 Aquecimento — perguntas que já funcionam bem
Faça estas perguntas no chat. Clique em **Mostrar código** para ver o SQL que o Genie gerou.
```
Quantos clientes temos por região?
Qual a receita total de 2025?
Quais são os planos mais caros?
Mostre a quantidade de atendimentos por canal em um gráfico de barras.
```
✔️ Perguntas "de contagem e soma" são fáceis. Agora vamos às perguntas que **dependem de regra de negócio**.

---
## 🧪 Rodada 1 · NPS
### ❓ Pergunte ao Genie (chat novo)
```
Qual o NPS por canal de atendimento?
```
### 🔍 O que costuma acontecer *sem contexto*
O Genie vê a coluna `nota_nps` e calcula a **média** das notas (ex.: URA ≈ 6,0). Parece razoável… mas **NPS não é média**. Veja a diferença:

In [0]:
%sql
-- ❌ ANTES: o que o Genie tende a fazer sem instrução
SELECT canal_atendimento, round(avg(nota_nps), 2) AS nps
FROM atendimentos
GROUP BY canal_atendimento ORDER BY nps DESC

In [0]:
%sql
-- ✅ DEPOIS: a definição oficial de NPS (% promotores - % detratores)
SELECT canal_atendimento,
       round(100 * (count_if(nota_nps >= 9) - count_if(nota_nps <= 6)) / count(*), 1) AS nps,
       round(100 * count_if(nota_nps >= 9) / count(*), 1) AS pct_promotores,
       round(100 * count_if(nota_nps <= 6) / count(*), 1) AS pct_detratores
FROM atendimentos
GROUP BY canal_atendimento ORDER BY nps DESC

In [0]:
prompt("""SELECT canal_atendimento,
       round(100 * (count_if(nota_nps >= 9) - count_if(nota_nps <= 6)) / count(*), 1) AS nps,
       round(100 * count_if(nota_nps >= 9) / count(*), 1) AS pct_promotores,
       round(100 * count_if(nota_nps <= 6) / count(*), 1) AS pct_detratores
FROM {T}.atendimentos
GROUP BY canal_atendimento ORDER BY nps DESC""", "📋 SQL com o nome completo das suas tabelas — use para registrar no Genie (Consultas SQL / Benchmarks)")

💬 *A média diz que a URA é "nota 6, quase ok". O NPS correto mostra que a URA tem **mais detratores do que promotores** (NPS negativo!). Decisões completamente diferentes.*

### 🧑‍🏫 Ensine o Genie
**Configurar → Instruções → Texto** — cole:
```
NPS (Net Promoter Score): calcule SEMPRE como (percentual de notas 9 e 10) menos (percentual de notas de 0 a 6), multiplicado por 100, usando a coluna nota_nps da tabela atendimentos. Notas 7 e 8 são neutras. NUNCA use a média de nota_nps para responder sobre NPS. O resultado varia de -100 a +100.
```
**Configurar → Instruções → Consultas SQL** → *Adicionar* — pergunta: `Qual o NPS por canal de atendimento?` — SQL: copie da caixa 📋 acima (já com o nome completo das suas tabelas).

### ✅ Pergunte de novo (chat novo)
```
Qual o NPS por canal de atendimento?
Qual foi o NPS mensal do WhatsApp em 2026?
O NPS é diferente quando o problema é resolvido no primeiro contato?
```
Repare que as **variações** da pergunta também passam a usar a regra certa — o Genie generaliza a partir da instrução.

---
## 🧪 Rodada 2 · Churn
### ❓ Pergunte ao Genie
```
Qual a taxa de churn do Pós-pago?
```
### 🔍 Sem contexto
O caminho "óbvio" é contar contratos com status *Cancelado* na tabela `contratos` e dividir pelo total — o que dá um número **acumulado de quase 2 anos**, que não é a taxa mensal que a diretoria acompanha:

In [0]:
%sql
-- ❌ ANTES: "churn" = % de contratos cancelados no cadastro (acumulado, sem noção de mês)
SELECT p.categoria,
       round(100 * count_if(c.status_contrato = 'Cancelado') / count(*), 2) AS taxa_churn_pct
FROM contratos c JOIN planos p USING (plano_id)
WHERE p.categoria = 'Pós-pago'
GROUP BY p.categoria

In [0]:
%sql
-- ✅ DEPOIS: churn mensal oficial = cancelados no mês / ativos no início do mês (tabela base_mensal)
SELECT b.mes_referencia,
       count_if(b.ativo_inicio_mes) AS base_ativa_inicio,
       count_if(b.cancelado_no_mes) AS cancelamentos,
       round(100 * count_if(b.cancelado_no_mes) / count_if(b.ativo_inicio_mes), 2) AS churn_mensal_pct
FROM base_mensal b JOIN planos p USING (plano_id)
WHERE p.categoria = 'Pós-pago'
GROUP BY b.mes_referencia ORDER BY b.mes_referencia

In [0]:
prompt("""SELECT b.mes_referencia,
       count_if(b.ativo_inicio_mes) AS base_ativa_inicio,
       count_if(b.cancelado_no_mes) AS cancelamentos,
       round(100 * count_if(b.cancelado_no_mes) / count_if(b.ativo_inicio_mes), 2) AS churn_mensal_pct
FROM {T}.base_mensal b JOIN {T}.planos p USING (plano_id)
WHERE p.categoria = 'Pós-pago'
GROUP BY b.mes_referencia ORDER BY b.mes_referencia""", "📋 SQL com o nome completo das suas tabelas — use para registrar no Genie (Consultas SQL / Benchmarks)")

### 🧑‍🏫 Ensine o Genie
**Instrução em texto:**
```
Churn: quando perguntarem sobre churn, taxa de cancelamento ou evasão, calcule a TAXA MENSAL usando a tabela base_mensal: número de contratos com cancelado_no_mes = true dividido pelo número de contratos com ativo_inicio_mes = true, no mesmo mes_referencia, em percentual. Se a pergunta for sobre um período (ano, trimestre), mostre a média das taxas mensais e também a evolução mês a mês. Não use status_contrato da tabela contratos para calcular churn.
```
**Consulta SQL de exemplo** — pergunta: `Qual o churn mensal por categoria de plano?` — SQL: copie da caixa 📋 abaixo.

In [0]:
prompt("""SELECT b.mes_referencia, p.categoria,
       round(100 * count_if(b.cancelado_no_mes) / count_if(b.ativo_inicio_mes), 2) AS churn_mensal_pct
FROM {T}.base_mensal b
JOIN {T}.planos p USING (plano_id)
GROUP BY ALL
ORDER BY 1, 2""", "📋 Consulta SQL de exemplo (Rodada 2)")

### ✅ Pergunte de novo
```
Qual a taxa de churn do Pós-pago?
O churn aumentou depois do reajuste de março de 2026?
Qual região tem o maior churn em 2026?
```

---
## 🧪 Rodada 3 · ARPU (receita média por cliente)
### ❓ Pergunte
```
Qual o ARPU médio por mês em 2026?
```
### 🔍 Sem contexto
É comum o Genie calcular a média do valor de **cada fatura**. Mas um cliente pode ter 2 contratos (móvel + fibra) — a TelcoBR mede ARPU **por cliente**:

In [0]:
%sql
-- ❌ ANTES: média por fatura/contrato
SELECT mes_referencia, round(avg(valor_faturado), 2) AS arpu
FROM faturamento_mensal
WHERE year(mes_referencia) = 2026
GROUP BY ALL ORDER BY 1

In [0]:
%sql
-- ✅ DEPOIS: receita do mês / clientes distintos faturados no mês
SELECT mes_referencia, round(sum(valor_faturado) / count(DISTINCT cliente_id), 2) AS arpu
FROM faturamento_mensal
WHERE year(mes_referencia) = 2026
GROUP BY ALL ORDER BY 1

In [0]:
prompt("""SELECT mes_referencia, round(sum(valor_faturado) / count(DISTINCT cliente_id), 2) AS arpu
FROM {T}.faturamento_mensal
WHERE year(mes_referencia) = 2026
GROUP BY ALL ORDER BY 1""", "📋 SQL com o nome completo das suas tabelas — use para registrar no Genie (Consultas SQL / Benchmarks)")

### 🧑‍🏫 Ensine o Genie — desta vez com uma **Expressão SQL** (medida reutilizável)
**Configurar → Instruções → Expressões SQL → Medida**
- **Nome:** `ARPU`
- **Código:** `SUM(faturamento_mensal.valor_faturado) / COUNT(DISTINCT faturamento_mensal.cliente_id)`
- **Sinônimos:** `receita média por cliente, ticket médio do cliente, ARPU`
- **Instrução:** `ARPU é sempre calculado por cliente distinto no mês, nunca pela média das faturas.`

Aproveite e crie também:
| Tipo | Nome | Código |
|---|---|---|
| Medida | `Receita` | `SUM(faturamento_mensal.valor_faturado)` |
| Filtro | `Pós-reajuste` | `faturamento_mensal.mes_referencia >= '2026-03-01'` |
| Dimensão | `Faixa de score` | `CASE WHEN clientes.score_credito < 500 THEN 'Alto risco' WHEN clientes.score_credito < 700 THEN 'Médio risco' ELSE 'Baixo risco' END` |

### ✅ Pergunte de novo
```
Qual o ARPU médio por mês em 2026?
Qual o ARPU por segmento de cliente no pós-reajuste?
```

---
## 🧪 Rodada 4 · Inadimplência
### ❓ Pergunte
```
Qual a taxa de inadimplência por faixa de score de crédito?
```
### 🔍 Sem contexto
"Inadimplente" pode ser qualquer fatura com atraso > 0, contado por **quantidade** de faturas. Na TelcoBR é **por valor** e só conta atraso **acima de 30 dias**:

In [0]:
%sql
-- ❌ ANTES: % de faturas com qualquer atraso
SELECT CASE WHEN c.score_credito < 500 THEN '1. Alto risco' WHEN c.score_credito < 700 THEN '2. Médio risco' ELSE '3. Baixo risco' END AS faixa_score,
       round(100 * count_if(f.dias_atraso > 0) / count(*), 2) AS inadimplencia_pct
FROM faturamento_mensal f JOIN clientes c USING (cliente_id)
GROUP BY ALL ORDER BY 1

In [0]:
%sql
-- ✅ DEPOIS: valor com atraso > 30 dias / valor faturado total (somente faturas pós-pagas: Pré-pago não tem fatura)
SELECT CASE WHEN c.score_credito < 500 THEN '1. Alto risco' WHEN c.score_credito < 700 THEN '2. Médio risco' ELSE '3. Baixo risco' END AS faixa_score,
       round(100 * sum(f.valor_faturado) FILTER (WHERE f.dias_atraso > 30) / sum(f.valor_faturado), 2) AS inadimplencia_pct
FROM faturamento_mensal f
JOIN clientes c USING (cliente_id)
JOIN planos p USING (plano_id)
WHERE p.categoria <> 'Pré-pago'
GROUP BY ALL ORDER BY 1

In [0]:
prompt("""SELECT CASE WHEN c.score_credito < 500 THEN '1. Alto risco' WHEN c.score_credito < 700 THEN '2. Médio risco' ELSE '3. Baixo risco' END AS faixa_score,
       round(100 * sum(f.valor_faturado) FILTER (WHERE f.dias_atraso > 30) / sum(f.valor_faturado), 2) AS inadimplencia_pct
FROM {T}.faturamento_mensal f
JOIN {T}.clientes c USING (cliente_id)
JOIN {T}.planos p USING (plano_id)
WHERE p.categoria <> 'Pré-pago'
GROUP BY ALL ORDER BY 1""", "📋 SQL com o nome completo das suas tabelas — use para registrar no Genie (Consultas SQL / Benchmarks)")

### 🧑‍🏫 Ensine o Genie
```
Inadimplência: é calculada por VALOR, não por quantidade. Taxa de inadimplência = soma de valor_faturado das faturas com dias_atraso > 30 dividida pela soma de valor_faturado total, em percentual. Exclua planos da categoria Pré-pago (não possuem fatura). Atrasos de 1 a 30 dias são "atraso", não inadimplência.
```
### ✅ Pergunte de novo
```
Qual a taxa de inadimplência por faixa de score de crédito?
Quanto temos de receita inadimplente por região em 2026?
```

---
## 🧪 Rodada 5 · O "dialeto" da empresa
Toda empresa tem jargões. Pergunte **antes** de ensinar:
```
Quantos clientes de alto risco temos na base ativa do Nordeste?
Como foi o gross add de Fibra no último trimestre?
Compare o pós-reajuste com o mesmo período do ano anterior em receita.
```
Provavelmente o Genie vai **perguntar de volta**, **chutar** um critério ou ignorar o termo. Agora cole esta instrução:
```
Glossário da TelcoBR:
- "Base ativa" = contratos com status_contrato = 'Ativo' (para fotos mensais, use ativo_inicio_mes na base_mensal). "Clientes ativos" = clientes com pelo menos um contrato Ativo.
- "Alto risco" = score_credito < 500; "Médio risco" = 500 a 699; "Baixo risco" = 700 ou mais.
- "Gross add" ou "ativações" = contratos com ativado_no_mes = true na base_mensal.
- "Reajuste" = aumento de 8,9% aplicado em março de 2026 aos planos Pós-pago, Controle, Fibra e Combo. "Pós-reajuste" = a partir de 01/03/2026; "pré-reajuste" = antes disso.
- "Móvel" = categorias Pré-pago, Controle e Pós-pago. "Residencial" = Fibra. Combo é tratado separadamente.
- Siglas de região: NE = Nordeste, SE = Sudeste, CO = Centro-Oeste, N = Norte, S = Sul.
- "Canal físico" = lojas com tipo_canal Loja Própria, Revenda ou Quiosque.
- "Último trimestre" = os 3 últimos meses completos disponíveis nos dados (os dados vão até set/2026).
Formato: valores monetários em R$ com 2 casas decimais; percentuais com 1 ou 2 casas. Responda sempre em português.
```
E repita as 3 perguntas acima. 🎉

In [0]:
%sql
-- ✅ Gabarito: clientes de alto risco na base ativa do Nordeste
SELECT count(DISTINCT c.cliente_id) AS clientes_alto_risco_ativos_ne
FROM clientes c
JOIN contratos k USING (cliente_id)
WHERE c.regiao = 'Nordeste' AND c.score_credito < 500 AND k.status_contrato = 'Ativo'

In [0]:
prompt("""SELECT count(DISTINCT c.cliente_id) AS clientes_alto_risco_ativos_ne
FROM {T}.clientes c
JOIN {T}.contratos k USING (cliente_id)
WHERE c.regiao = 'Nordeste' AND c.score_credito < 500 AND k.status_contrato = 'Ativo'""", "📋 SQL com o nome completo das suas tabelas — use para registrar no Genie (Consultas SQL / Benchmarks)")

---
## 🎯 Rodada 6 · Respostas determinísticas com *Consultas Confiáveis* (parâmetros)
Para perguntas que a diretoria faz **toda semana**, queremos a **mesma** resposta, sempre — sem "criatividade".
Registre uma consulta SQL **com parâmetros**. Quando o Genie usa uma consulta registrada, a resposta aparece marcada como **Confiável (Trusted)** ✅.

**Configurar → Instruções → Consultas SQL → Adicionar**
- **Pergunta:** `Qual o resumo de KPIs de um mês?` — **SQL:** copie da caixa 📋 abaixo.

In [0]:
prompt("""WITH fat AS (
  SELECT sum(valor_faturado) AS receita,
         count(DISTINCT cliente_id) AS clientes_faturados,
         sum(valor_faturado) FILTER (WHERE dias_atraso > 30) AS receita_inadimplente
  FROM {T}.faturamento_mensal
  WHERE mes_referencia = date_trunc('MONTH', :mes)::date
), base AS (
  SELECT count_if(ativo_inicio_mes) AS base_ativa, count_if(cancelado_no_mes) AS cancelamentos, count_if(ativado_no_mes) AS ativacoes
  FROM {T}.base_mensal
  WHERE mes_referencia = date_trunc('MONTH', :mes)::date
), sac AS (
  SELECT round(100 * (count_if(nota_nps >= 9) - count_if(nota_nps <= 6)) / count(*), 1) AS nps
  FROM {T}.atendimentos
  WHERE date_trunc('MONTH', data_abertura) = date_trunc('MONTH', :mes)
)
SELECT round(receita, 2) AS receita_rs, base_ativa, ativacoes, cancelamentos,
       round(100 * cancelamentos / base_ativa, 2) AS churn_pct,
       round(receita / clientes_faturados, 2) AS arpu_rs,
       nps
FROM fat, base, sac""", "📋 Consulta SQL parametrizada (Rodada 6)")

- Defina o parâmetro **`mes`** como tipo **Data**.

### ✅ Teste
```
Me dê o resumo de KPIs de abril de 2026.
E de abril de 2025?
```
Pergunte 3 vezes, em chats diferentes: o resultado é **idêntico** e vem com o selo **Confiável**.

In [0]:
%sql
-- Conferência do resumo de KPIs (abr/2026)
DECLARE OR REPLACE VARIABLE mes DATE DEFAULT DATE'2026-04-01';
WITH fat AS (
  SELECT sum(valor_faturado) AS receita, count(DISTINCT cliente_id) AS clientes_faturados
  FROM faturamento_mensal WHERE mes_referencia = mes
), base AS (
  SELECT count_if(ativo_inicio_mes) AS base_ativa, count_if(cancelado_no_mes) AS cancelamentos, count_if(ativado_no_mes) AS ativacoes
  FROM base_mensal WHERE mes_referencia = mes
), sac AS (
  SELECT round(100 * (count_if(nota_nps >= 9) - count_if(nota_nps <= 6)) / count(*), 1) AS nps
  FROM atendimentos WHERE date_trunc('MONTH', data_abertura) = mes
)
SELECT round(receita, 2) AS receita_rs, base_ativa, ativacoes, cancelamentos,
       round(100 * cancelamentos / base_ativa, 2) AS churn_pct, round(receita / clientes_faturados, 2) AS arpu_rs, nps
FROM fat, base, sac

---
## 🔗 Bônus · Relacionamentos (Joins)
Em **Configurar → Instruções → Joins**, confira (ou adicione) como as tabelas se ligam. Isso evita que o Genie "invente" relacionamentos:
| Tabela | Coluna | Liga com | Tipo |
|---|---|---|---|
| `contratos` | `cliente_id` | `clientes.cliente_id` | muitos para um |
| `contratos` | `plano_id` | `planos.plano_id` | muitos para um |
| `contratos` | `loja_id` | `lojas.loja_id` | muitos para um |
| `base_mensal` / `faturamento_mensal` | `contrato_id` | `contratos.contrato_id` | muitos para um |
| `atendimentos` | `cliente_id` | `clientes.cliente_id` | muitos para um |

---
## 📏 Passo final · Benchmarks: provando a efetividade com números
Na aba **Benchmarks**, adicione as perguntas abaixo **com a resposta SQL correta** (as consultas ✅ deste notebook). Depois clique em **Executar benchmarks**.

| # | Pergunta do benchmark | SQL de referência |
|---|---|---|
| 1 | Qual o NPS por canal de atendimento? | Rodada 1 ✅ |
| 2 | Qual o churn mensal do Pós-pago? | Rodada 2 ✅ |
| 3 | Qual o ARPU por mês em 2026? | Rodada 3 ✅ |
| 4 | Qual a inadimplência por faixa de score? | Rodada 4 ✅ |
| 5 | Quantos clientes de alto risco temos na base ativa do NE? | Rodada 5 ✅ |

💡 **Para uma demo de impacto:** o facilitador roda os benchmarks **antes** de colar as instruções (normalmente 1–2 de 5 corretas) e **depois** (5 de 5). É a prova numérica de que o conhecimento do negócio faz a diferença.

### 🛟 Plano B · Genie Agent gabarito
Rode o notebook **`06_gabaritos_plano_b`**: ele cria na sua conta um Genie Agent já configurado com todas as instruções, consultas de exemplo (incl. a parametrizada), expressões SQL, joins e benchmarks.

### 💬 Para discutir
- Quem na sua área deveria ser o "dono" das instruções de cada KPI?
- Cada Genie Agent tem suas próprias instruções. E se a definição de churn mudar — vamos atualizar 20 agentes? 🤔 → No notebook **05** vemos como colocar a definição **uma vez** na camada semântica (Metric Views + Glossário) e todos os agentes, dashboards e o **Genie One** passam a usá-la.